## RAG Document Question-Answering System

A **Retrieval-Augmented Generation (RAG)** pipeline that answers questions from a custom document (PDF/notes/resume/research paper) by:

1. **Ingesting** the document and splitting it into chunks
2. **Embedding** each chunk into a vector
3. **Storing** the vectors in a vector database for similarity search
4. **Retrieving** the chunks most relevant to a user's question
5. **Generating** a grounded answer using an LLM conditioned on the retrieved context



## 1. Setup — Install & Import Dependencies

In [6]:
!pip install pypdf sentence-transformers faiss-cpu transformers scikit-learn torch -q

import os
import re
import numpy as np
import pypdf

print("Environment ready.")

Environment ready.


## 2. Document Ingestion

In [7]:
def create_demo_pdf(path="sample_notes.pdf"):
    """Generates a small demo PDF of ML/RAG study notes so this notebook is self-contained.
    Skip this cell and just set PDF_PATH to your own file if you already have one."""
    from reportlab.lib.pagesizes import LETTER
    from reportlab.platypus import SimpleDocTemplate, Paragraph
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.units import inch

    doc = SimpleDocTemplate(path, pagesize=LETTER, topMargin=0.8*inch, bottomMargin=0.8*inch)
    styles = getSampleStyleSheet()
    title_style = ParagraphStyle("T", parent=styles["Heading1"], spaceAfter=14)
    h2 = ParagraphStyle("H2", parent=styles["Heading2"], spaceBefore=12, spaceAfter=6)
    body = ParagraphStyle("Body", parent=styles["Normal"], spaceAfter=10, leading=15)

    sections = [
        ("What is Machine Learning?",
         "Machine Learning (ML) is a subfield of Artificial Intelligence that enables computers to "
         "learn patterns from data without being explicitly programmed. Instead of writing fixed "
         "rules, an ML model is trained on examples and gradually improves its performance on a "
         "task, such as classification or prediction, by minimizing an error metric during training."),
        ("Types of Machine Learning",
         "There are three main types of machine learning. Supervised learning uses labeled data, "
         "where each input has a known output, to train models such as regression and classification "
         "algorithms. Unsupervised learning works with unlabeled data to discover hidden patterns, "
         "such as clustering and dimensionality reduction. Reinforcement learning trains an agent to "
         "make sequential decisions by rewarding desirable actions and penalizing undesirable ones."),
        ("Neural Networks",
         "A neural network is a computational model inspired by the human brain, composed of layers "
         "of interconnected nodes called neurons. Each connection has a weight that is adjusted "
         "during training through backpropagation, which uses gradient descent to minimize a loss "
         "function. Deep learning refers to neural networks with many hidden layers, which allow the "
         "model to learn increasingly abstract representations of the input data."),
        ("Overfitting and Regularization",
         "Overfitting occurs when a model learns the training data too well, including its noise, "
         "and performs poorly on new, unseen data. Common techniques to reduce overfitting include "
         "regularization methods such as L1 and L2 penalties, dropout layers in neural networks, "
         "early stopping during training, and increasing the size or diversity of the training data."),
        ("What is Retrieval-Augmented Generation (RAG)?",
         "Retrieval-Augmented Generation is a technique that combines a retrieval system with a "
         "generative language model. Instead of relying only on knowledge stored in the model's "
         "parameters, RAG retrieves relevant text chunks from an external document collection using "
         "vector similarity search, and feeds those chunks as context to the language model so it "
         "can generate an answer grounded in the retrieved evidence. This reduces hallucination and "
         "allows question answering over private or domain-specific documents."),
        ("Vector Embeddings",
         "A vector embedding is a numerical representation of text that captures its semantic "
         "meaning in a high-dimensional space. Sentences with similar meaning are mapped to nearby "
         "points in this space. Embedding models, often built on transformer architectures, convert "
         "text chunks and user queries into embeddings so that similarity search, typically using "
         "cosine similarity, can identify the most relevant chunks for a query."),
        ("Evaluation Metrics for Classification",
         "Common evaluation metrics for classification models include accuracy, precision, recall, "
         "and F1-score. Accuracy measures the overall proportion of correct predictions. Precision "
         "measures how many predicted positives are actually correct, while recall measures how many "
         "actual positives were correctly identified. The F1-score is the harmonic mean of precision "
         "and recall, and is especially useful when classes are imbalanced."),
    ]

    content = [Paragraph("Machine Learning and AI: Study Notes", title_style)]
    for heading, text in sections:
        content.append(Paragraph(heading, h2))
        content.append(Paragraph(text, body))
    doc.build(content)
    return path

PDF_PATH = "sample_notes.pdf"
print(f"Using document: {PDF_PATH}")

Using document: sample_notes.pdf


In [8]:
def load_pdf(path: str) -> str:
    """Extracts raw text from a PDF file."""
    reader = pypdf.PdfReader(path)
    text = ""
    for page in reader.pages:
        text += (page.extract_text() or "") + "\n"
    return text

raw_text = load_pdf(PDF_PATH)
print(f"Extracted {len(raw_text)} characters from the document.\n")
print(raw_text[:400], "...")

Extracted 3179 characters from the document.

Machine Learning and AI: Study Notes
What is Machine Learning?
Machine Learning (ML) is a subfield of Artificial Intelligence that enables computers to learn patterns
from data without being explicitly programmed. Instead of writing fixed rules, an ML model is trained on
examples and gradually improves its performance on a task, such as classification or prediction, by
minimizing an error metric d ...


## 3. Text Chunking

In [9]:
def chunk_text(text: str, chunk_size: int = 120, overlap: int = 30) -> list:
    """Splits text into overlapping chunks of `chunk_size` words."""
    text = re.sub(r"\s+", " ", text).strip()
    words = text.split(" ")
    chunks, start = [], 0
    while start < len(words):
        end = start + chunk_size
        chunks.append(" ".join(words[start:end]))
        start += chunk_size - overlap
    return chunks

chunks = chunk_text(raw_text, chunk_size=120, overlap=30)
print(f"Document split into {len(chunks)} chunks.\n")
for i, c in enumerate(chunks):
    print(f"[{i}] {c[:90]}...")

Document split into 6 chunks.

[0] Machine Learning and AI: Study Notes What is Machine Learning? Machine Learning (ML) is a ...
[1] such as regression and classification algorithms. Unsupervised learning works with unlabel...
[2] hidden layers, which allow the model to learn increasingly abstract representations of the...
[3] language model. Instead of relying only on knowledge stored in the model's parameters, RAG...
[4] this space. Embedding models, often built on transformer architectures, convert text chunk...
[5] useful when classes are imbalanced....


## 4. Embedding Creation

In [10]:
class EmbeddingModel:
    """Wraps an embedding backend. Tries sentence-transformers, falls back to TF-IDF."""

    def __init__(self):
        self.backend = None
        try:
            from sentence_transformers import SentenceTransformer
            self.model = SentenceTransformer("all-MiniLM-L6-v2")
            self.backend = "sentence-transformers"
        except Exception:
            from sklearn.feature_extraction.text import TfidfVectorizer
            self.vectorizer = TfidfVectorizer(stop_words="english")
            self.backend = "tfidf"
        print(f"[EmbeddingModel] using backend: {self.backend}")

    def fit(self, texts):
        """Fit on the corpus (chunks) and return their embeddings."""
        if self.backend == "tfidf":
            return self.vectorizer.fit_transform(texts).toarray()
        return self.model.encode(texts, show_progress_bar=False)

    def encode(self, texts):
        """Embed new text (e.g. a query) using the already-fitted vocabulary/model."""
        if self.backend == "tfidf":
            return self.vectorizer.transform(texts).toarray()
        return self.model.encode(texts, show_progress_bar=False)

embedder = EmbeddingModel()
chunk_embeddings = embedder.fit(chunks)
print("Embedding matrix shape:", np.array(chunk_embeddings).shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[EmbeddingModel] using backend: sentence-transformers
Embedding matrix shape: (6, 384)


## 5. Vector Database

In [11]:
class VectorStore:
    """Stores chunk embeddings and supports similarity search. FAISS with a NumPy fallback."""

    def __init__(self, embeddings, chunks):
        self.chunks = chunks
        self.embeddings = np.array(embeddings)
        self.backend = None
        try:
            import faiss
            dim = self.embeddings.shape[1]
            self.index = faiss.IndexFlatIP(dim)
            norm = self.embeddings / (np.linalg.norm(self.embeddings, axis=1, keepdims=True) + 1e-10)
            self.index.add(norm.astype("float32"))
            self.backend = "faiss"
        except Exception:
            self.backend = "numpy"
        print(f"[VectorStore] using backend: {self.backend}")

    def search(self, query_vec, top_k=3):
        query_vec = np.array(query_vec).reshape(1, -1)
        if self.backend == "faiss":
            qn = query_vec / (np.linalg.norm(query_vec) + 1e-10)
            scores, idx = self.index.search(qn.astype("float32"), top_k)
            return [(self.chunks[i], float(scores[0][r])) for r, i in enumerate(idx[0])]
        else:
            from sklearn.metrics.pairwise import cosine_similarity
            sims = cosine_similarity(query_vec, self.embeddings)[0]
            top_idx = np.argsort(sims)[::-1][:top_k]
            return [(self.chunks[i], float(sims[i])) for i in top_idx]

vector_store = VectorStore(chunk_embeddings, chunks)

[VectorStore] using backend: faiss


## 6. Query Processing & Context Retrieval

In [12]:
def retrieve(query: str, top_k: int = 3):
    q_vec = embedder.encode([query])[0]
    return vector_store.search(q_vec, top_k=top_k)

demo_query = "What is Retrieval-Augmented Generation?"
retrieved = retrieve(demo_query, top_k=3)

print(f"Query: {demo_query}\n")
for rank, (chunk, score) in enumerate(retrieved, 1):
    print(f"#{rank} (score={score:.3f}): {chunk[:150]}...\n")

Query: What is Retrieval-Augmented Generation?

#1 (score=0.631): hidden layers, which allow the model to learn increasingly abstract representations of the input data. Overfitting and Regularization Overfitting occu...

#2 (score=0.319): language model. Instead of relying only on knowledge stored in the model's parameters, RAG retrieves relevant text chunks from an external document co...

#3 (score=0.312): this space. Embedding models, often built on transformer architectures, convert text chunks and user queries into embeddings so that similarity search...



## 7. Answer Generation

In [13]:
class AnswerGenerator:
    """Generates an answer from retrieved context. flan-t5 with an extractive fallback."""

    def __init__(self):
        self.backend = None
        try:
            from transformers import pipeline
            self.pipe = pipeline("text2text-generation", model="google/flan-t5-base")
            self.backend = "flan-t5"
        except Exception:
            self.backend = "extractive"
        print(f"[AnswerGenerator] using backend: {self.backend}")

    def generate(self, query: str, context_chunks: list) -> str:
        context = "\n".join(context_chunks)
        if self.backend == "flan-t5":
            prompt = (
                f"Answer the question using only the context below.\n"
                f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
            )
            out = self.pipe(prompt, max_new_tokens=150)[0]["generated_text"]
            return out.strip()
        return self._extractive(query, context_chunks)

    def _extractive(self, query: str, context_chunks: list) -> str:
        """Fallback: return the sentence with the highest word overlap with the query."""
        q_words = set(re.findall(r"\w+", query.lower()))
        best_sentence, best_score = "", -1
        for chunk in context_chunks:
            for sent in re.split(r"(?<=[.!?])\s+", chunk):
                s_words = set(re.findall(r"\w+", sent.lower()))
                overlap = len(q_words & s_words)
                if overlap > best_score:
                    best_score, best_sentence = overlap, sent
        return best_sentence.strip() if best_sentence else "I could not find relevant information in the document."

generator = AnswerGenerator()

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

[AnswerGenerator] using backend: extractive


## 8. Putting It All Together — the RAGPipeline Class

In [14]:
class RAGPipeline:
    def __init__(self, chunk_size=120, overlap=30, top_k=3):
        self.chunk_size = chunk_size
        self.overlap = overlap
        self.top_k = top_k
        self.embedder = EmbeddingModel()
        self.generator = AnswerGenerator()
        self.store = None
        self.chunks = None

    def _load_pdf(self, path):
        reader = pypdf.PdfReader(path)
        return "\n".join(page.extract_text() or "" for page in reader.pages)

    def _chunk_text(self, text):
        text = re.sub(r"\s+", " ", text).strip()
        words = text.split(" ")
        out, start = [], 0
        while start < len(words):
            end = start + self.chunk_size
            out.append(" ".join(words[start:end]))
            start += self.chunk_size - self.overlap
        return out

    def build_index(self, pdf_path: str):
        raw = self._load_pdf(pdf_path)
        self.chunks = self._chunk_text(raw)
        embeddings = self.embedder.fit(self.chunks)
        self.store = VectorStore(embeddings, self.chunks)
        print(f"Indexed {len(self.chunks)} chunks from '{pdf_path}'.")

    def ask(self, query: str) -> dict:
        if self.store is None:
            raise RuntimeError("Call build_index() before ask().")
        q_vec = self.embedder.encode([query])[0]
        results = self.store.search(q_vec, top_k=self.top_k)
        context_chunks = [c for c, _ in results]
        answer = self.generator.generate(query, context_chunks)
        return {"question": query, "answer": answer, "sources": results}

## 9. Demo: Asking Questions About the Document

In [15]:
rag = RAGPipeline(chunk_size=120, overlap=30, top_k=3)
rag.build_index(PDF_PATH)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[EmbeddingModel] using backend: sentence-transformers
[AnswerGenerator] using backend: extractive
[VectorStore] using backend: faiss
Indexed 6 chunks from 'sample_notes.pdf'.


In [16]:
questions = [
    "What is Retrieval-Augmented Generation?",
    "What causes overfitting and how can it be reduced?",
    "What is the difference between supervised and unsupervised learning?",
    "What is a vector embedding?",
]

for q in questions:
    result = rag.ask(q)
    print("Q:", result["question"])
    print("A:", result["answer"])
    top_chunk, top_score = result["sources"][0]
    print(f"   (top source, score={top_score:.3f}): {top_chunk[:120]}...")
    print("-" * 90)

Q: What is Retrieval-Augmented Generation?
A: What is Retrieval-Augmented Generation (RAG)?
   (top source, score=0.631): hidden layers, which allow the model to learn increasingly abstract representations of the input data. Overfitting and R...
------------------------------------------------------------------------------------------
Q: What causes overfitting and how can it be reduced?
A: Overfitting and Regularization Overfitting occurs when a model learns the training data too well, including its noise, and performs poorly on new, unseen data.
   (top source, score=0.277): hidden layers, which allow the model to learn increasingly abstract representations of the input data. Overfitting and R...
------------------------------------------------------------------------------------------
Q: What is the difference between supervised and unsupervised learning?
A: Machine Learning and AI: Study Notes What is Machine Learning?
   (top source, score=0.571): Machine Learning and AI: Study No

In [17]:
your_question = "What is machine learning?"

result = rag.ask(your_question)
print("Q:", result["question"])
print("A:", result["answer"])
print("\nRetrieved context used to answer:")
for i, (chunk, score) in enumerate(result["sources"], 1):
    print(f"\n[{i}] score={score:.3f}\n{chunk}")

Q: What is machine learning?
A: Machine Learning and AI: Study Notes What is Machine Learning?

Retrieved context used to answer:

[1] score=0.756
Machine Learning and AI: Study Notes What is Machine Learning? Machine Learning (ML) is a subfield of Artificial Intelligence that enables computers to learn patterns from data without being explicitly programmed. Instead of writing fixed rules, an ML model is trained on examples and gradually improves its performance on a task, such as classification or prediction, by minimizing an error metric during training. Types of Machine Learning There are three main types of machine learning. Supervised learning uses labeled data, where each input has a known output, to train models such as regression and classification algorithms. Unsupervised learning works with unlabeled data to discover hidden patterns, such as clustering and dimensionality reduction. Reinforcement learning trains an agent to make sequential

[2] score=0.523
such as regression a

## Conclusion

This notebook implements the full RAG pipeline end to end: **ingest → chunk → embed →
store → retrieve → generate**. It automatically uses stronger models
(`sentence-transformers`, `FAISS`, `flan-t5`) when available, and degrades gracefully to
lightweight offline alternatives (TF-IDF, NumPy similarity, extractive answering)
otherwise — so it runs the same way in a restricted sandbox, in Google Colab, or on a
local machine with a GPU.

RAG systems like this one power real-world chatbots, knowledge assistants, enterprise
search, and AI documentation tools by grounding LLM answers in private, up-to-date data
instead of only the model's frozen training knowledge.